# Viscan OPG - architecture benchmark: Faster R-CNN vs RetinaNet, TTA, and WBF

One Kaggle GPU session that measures every comparison this repository claims:

1. Faster R-CNN ResNet-50 FPN V2 (deployed checkpoint, downloaded and SHA-256 verified)
2. Faster R-CNN + horizontal-flip TTA
3. RetinaNet ResNet-50 FPN V2 (trained here on the identical split, seed, schedule and 800x1333 scale)
4. RetinaNet + hflip TTA
5. Weighted Box Fusion of the two plain models
6. Weighted Box Fusion of the two TTA models

**Protocol**
- Data: DENTEX pinned revision, `SEED=42` split of 635 train / 70 val / 50 held-out test.
- Training: COCO_V1 fine-tuning, 12 epochs, SGD lr 0.005, cosine schedule, AMP, batch 2, flip + mild colour jitter.
- The operating confidence threshold is selected **only** on the 70-image validation split (macro-F1 sweep
  0.10-0.90), then applied once to the 50-image test split.
- Metrics: COCO AP / AP50 / AP75 / AR@100 overall and per class, per-class P/R/F1, 5x5 confusion matrix,
  failure cases. Fusion uses WBF at IoU 0.55.

**Before running:** Settings -> Accelerator **T4 x2**, Settings -> Internet **On**. Training RetinaNet for
12 epochs takes multiple hours; Kaggle GPU sessions cap at 12 h and there is no resume logic.

**Honesty note:** no benchmark metric is pre-filled. Every measured row is read from JSON written by this run.
The DENTEX leaderboard table is external and labelled as such; its test set is different from ours.

**Licences:** torchvision code and both architectures BSD-3-Clause; DENTEX CC BY-NC-SA 4.0 (weights inherit the
non-commercial restriction). No AGPL component is used.


In [ ]:
import os, sys, json, time, platform, subprocess
from pathlib import Path

import torch, torchvision

print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("platform   :", platform.platform())

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> T4 x2, then re-run."
print("gpu        :", torch.cuda.get_device_name(0))
print("vram (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
REPO_URL = "https://github.com/surendra902/viscan-opg.git"
PINNED_COMMIT = "2cf40e240c6d23ed99209ad0f9a28576093a1c7b"
WORKDIR = Path("/kaggle/working/viscan-opg")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "--quiet", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORKDIR)], check=True)

subprocess.run(["git", "-C", str(WORKDIR), "checkout", "--quiet", PINNED_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(WORKDIR), "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == PINNED_COMMIT, f"checkout mismatch: {head}"
print("repo pinned at", head)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

for mod in ("yaml", "pycocotools", "matplotlib", "pandas", "PIL"):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)

import model as viscan_model
assert viscan_model.HAS_TORCHVISION
print("imports OK | classes:", viscan_model.CLASS_NAMES)


In [ ]:
os.chdir(WORKDIR)

data_dir = Path("data/dentex")
test_images = data_dir / "images" / "test"
ready = ((data_dir / "annotations" / "instances_test.json").exists()
         and test_images.is_dir() and any(test_images.iterdir()))

if ready:
    print("[SKIP] dataset already prepared at", data_dir.resolve())
else:
    print("[RUN] downloading + converting DENTEX (pinned revision) into", data_dir.resolve())
    proc = subprocess.run(["bash", "-lc",
        f'set -o pipefail; "{sys.executable}" prepare_data.py data/dentex 2>&1 | tee prepare_data.log'],
        check=False)
    assert proc.returncode == 0, "prepare_data.py failed - check prepare_data.log"


In [ ]:
os.chdir(WORKDIR)
import pandas as pd

data_dir = Path("data/dentex")
splits = ("train", "val", "test")
summary = {}

for s in splits:
    with open(data_dir / "annotations" / f"instances_{s}.json", encoding="utf-8") as f:
        coco = json.load(f)
    per_class = {cid: 0 for cid in viscan_model.CLASS_NAMES if cid != 0}
    for ann in coco["annotations"]:
        per_class[ann["category_id"]] += 1
    summary[s] = {
        "images": len(coco["images"]),
        "boxes": len(coco["annotations"]),
        "per_class": per_class,
        "files": {img["file_name"] for img in coco["images"]},
    }

rows = []
for s in splits:
    row = {"split": s, "images": summary[s]["images"], "boxes": summary[s]["boxes"]}
    for cid, name in viscan_model.CLASS_NAMES.items():
        if cid != 0:
            row[name] = summary[s]["per_class"][cid]
    rows.append(row)
display(pd.DataFrame(rows).set_index("split"))

assert summary["train"]["images"] == 635 and summary["val"]["images"] == 70
assert summary["test"]["images"] == 50
assert summary["train"]["boxes"] + summary["val"]["boxes"] == 3529
assert summary["test"]["boxes"] == 182
assert not (summary["train"]["files"] & summary["val"]["files"])
assert not (summary["train"]["files"] & summary["test"]["files"])
assert not (summary["val"]["files"] & summary["test"]["files"])

print("split verified: 635/70/50 images, 3,529 pooled boxes, 182 held-out test boxes, no filename overlap")


In [ ]:
os.chdir(WORKDIR)
import hashlib
from urllib.request import urlretrieve

WEIGHTS_URL = "https://huggingface.co/surendra08/viscan-opg-weights/resolve/main/best_model.pth"
WEIGHTS_SHA256 = "e0b49b12f48fc71ec1f583db08f107650659571e7c2b9cbd454dbfed13b46a1f"

ckpt = Path("checkpoints/best_model.pth")
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    print("[DOWNLOAD]", WEIGHTS_URL)
    urlretrieve(WEIGHTS_URL, ckpt)

digest = hashlib.sha256()
with open(ckpt, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        digest.update(chunk)
actual = digest.hexdigest()
assert actual == WEIGHTS_SHA256, f"checkpoint hash mismatch: {actual}"
print(f"Faster R-CNN checkpoint verified: {ckpt.stat().st_size:,} bytes | sha256 {actual}")


In [ ]:
os.chdir(WORKDIR)

# Faster R-CNN, no TTA. Dumps validation+test predictions for the fusion step.
cmd = (f'"{sys.executable}" evaluate.py '
       f'--weights checkpoints/best_model.pth --config configs/baseline.yaml '
       f'--data_dir data/dentex --output_dir evaluation_frcnn --skip_variant '
       f'--select_threshold val --dump_predictions preds/frcnn 2>&1 | tee eval_frcnn.log')
proc = subprocess.run(["bash", "-lc", f"set -o pipefail; {cmd}"], check=False)
assert proc.returncode == 0, "Faster R-CNN evaluation failed - inspect eval_frcnn.log"
print(Path("evaluation_frcnn/final_evaluation_report.json").read_text()[:400])


In [ ]:
os.chdir(WORKDIR)

cmd = (f'"{sys.executable}" evaluate.py '
       f'--weights checkpoints/best_model.pth --config configs/baseline.yaml '
       f'--data_dir data/dentex --output_dir evaluation_frcnn_tta --skip_variant '
       f'--select_threshold val --tta hflip --dump_predictions preds/frcnn_tta 2>&1 | tee eval_frcnn_tta.log')
proc = subprocess.run(["bash", "-lc", f"set -o pipefail; {cmd}"], check=False)
assert proc.returncode == 0, "Faster R-CNN TTA evaluation failed - inspect eval_frcnn_tta.log"
print("Faster R-CNN + hflip TTA done")


In [ ]:
os.chdir(WORKDIR)

train_start = time.time()
proc = subprocess.run(["bash", "-lc",
    f'set -o pipefail; "{sys.executable}" train.py --config configs/retinanet.yaml 2>&1 | tee train_retinanet.log'],
    check=False)
print(f"training wall time: {(time.time() - train_start) / 60.0:.1f} min")
assert proc.returncode == 0, "training failed - inspect train_retinanet.log"

ckpt = Path("checkpoints/retinanet/best_model.pth")
assert ckpt.exists(), "best RetinaNet checkpoint missing"
state = torch.load(ckpt, map_location="cpu", weights_only=False)
print("best epoch:", state["epoch"], "| best val mAP@50:", round(float(state["val_map50"]), 4))

for png in ("runs/retinanet/loss_curve.png", "runs/retinanet/val_ap_curve.png"):
    print(("OK   " if Path(png).exists() else "MISSING ") + png)
del state
torch.cuda.empty_cache()


In [ ]:
os.chdir(WORKDIR)

cmd = (f'"{sys.executable}" evaluate.py '
       f'--weights checkpoints/retinanet/best_model.pth --config configs/retinanet.yaml '
       f'--data_dir data/dentex --output_dir evaluation_retinanet --skip_variant '
       f'--select_threshold val --dump_predictions preds/retinanet 2>&1 | tee eval_retinanet.log')
proc = subprocess.run(["bash", "-lc", f"set -o pipefail; {cmd}"], check=False)
assert proc.returncode == 0, "RetinaNet evaluation failed - inspect eval_retinanet.log"
print("RetinaNet evaluation done")


In [ ]:
os.chdir(WORKDIR)

cmd = (f'"{sys.executable}" evaluate.py '
       f'--weights checkpoints/retinanet/best_model.pth --config configs/retinanet.yaml '
       f'--data_dir data/dentex --output_dir evaluation_retinanet_tta --skip_variant '
       f'--select_threshold val --tta hflip --dump_predictions preds/retinanet_tta 2>&1 | tee eval_retinanet_tta.log')
proc = subprocess.run(["bash", "-lc", f"set -o pipefail; {cmd}"], check=False)
assert proc.returncode == 0, "RetinaNet TTA evaluation failed - inspect eval_retinanet_tta.log"
print("RetinaNet + hflip TTA done")


In [ ]:
os.chdir(WORKDIR)

def run_ensemble(tag, a, b, out_dir):
    cmd = (f'"{sys.executable}" ensemble_wbf.py '
           f'--gt data/dentex/annotations/instances_test.json '
           f'--images_dir data/dentex/images/test '
           f'--preds {a}/predictions_test.json {b}/predictions_test.json '
           f'--val-gt data/dentex/annotations/instances_val.json '
           f'--val-preds {a}/predictions_val.json {b}/predictions_val.json '
           f'--output_dir {out_dir} --iou_thresh 0.55 2>&1 | tee eval_ensemble_{tag}.log')
    proc = subprocess.run(["bash", "-lc", f"set -o pipefail; {cmd}"], check=False)
    assert proc.returncode == 0, f"ensemble {tag} failed - inspect eval_ensemble_{tag}.log"
    print(f"ensemble {tag} done -> {out_dir}")

run_ensemble("plain", "preds/frcnn", "preds/retinanet", "evaluation_ensemble")
run_ensemble("tta", "preds/frcnn_tta", "preds/retinanet_tta", "evaluation_ensemble_tta")


In [ ]:
os.chdir(WORKDIR)
import numpy as np
import pandas as pd

RUNS = [
    ("Faster R-CNN", "evaluation_frcnn"),
    ("Faster R-CNN + hflip TTA", "evaluation_frcnn_tta"),
    ("RetinaNet", "evaluation_retinanet"),
    ("RetinaNet + hflip TTA", "evaluation_retinanet_tta"),
    ("WBF (Faster R-CNN + RetinaNet)", "evaluation_ensemble"),
    ("WBF (TTA pair)", "evaluation_ensemble_tta"),
]

reports = {}
for label, folder in RUNS:
    path = Path(folder) / "final_evaluation_report.json"
    assert path.exists(), f"missing report: {path}"
    reports[label] = json.loads(path.read_text(encoding="utf-8"))

rows, prf_rows = [], []
for label, _ in RUNS:
    rep = reports[label]
    overall = rep["coco_map"]["overall"]
    sel = rep["threshold_selection"]
    tau = sel.get("chosen_threshold", sel.get("best_threshold"))
    tm = rep["threshold_metrics"]
    latency = rep.get("cpu_latency_ms")
    rows.append({
        "run": label,
        "AP": overall["map_50_95"],
        "AP50": overall["map_50"],
        "AP75": overall["map_75"],
        "AR@100": overall.get("ar_100"),
        "tau": tau,
        "val macro-F1": sel.get("best_val_macro_f1"),
        "test macro-F1": tm["macro_f1"],
        "CPU ms/img": latency,
        "TTA": rep.get("tta", "n/a"),
    })
    for cls, vals in rep["coco_map"]["per_class"].items():
        prf_rows.append({"run": label, "class": cls, "AP50": vals["map_50"], "AP": vals["map_50_95"]})

summary = pd.DataFrame(rows).set_index("run")
print("Measured in this session (higher is better; CPU ms is single-pass and not comparable across machines):")
display(summary.round(4))

print("\nPer-class AP:")
display(pd.DataFrame(prf_rows).pivot(index="class", columns="run", values="AP50").round(4))

best_run = summary["AP50"].idxmax()
print(f"\nBest measured test AP50: {best_run} ({summary.loc[best_run, 'AP50']:.4f})")

print("\nSelected thresholds and validation macro-F1 per run:")
display(summary[["tau", "val macro-F1", "test macro-F1"]].round(4))

print("\nPer-class precision/recall/F1 at each run's selected threshold:")
for label, _ in RUNS:
    tm = reports[label]["threshold_metrics"]
    print(f"\n[{label}] tau={tm['confidence_threshold']:.2f} macro-F1={tm['macro_f1']}")
    display(pd.DataFrame([{"class": k, **v} for k, v in tm["class_metrics"].items()]).set_index("class"))

print("\nConfusion matrices (rows = ground truth, columns = prediction):")
labels = [viscan_model.CLASS_NAMES[c] for c in range(5)]
for label, _ in RUNS:
    cm = reports[label]["threshold_metrics"]["confusion_matrix"]
    print(f"\n[{label}]")
    display(pd.DataFrame(np.asarray(cm), index=labels, columns=labels))

from pathlib import Path as _P
for label, folder in RUNS:
    n = reports[label]["num_failure_cases_extracted"]
    print(f"{label}: {n} failure cases -> {folder}/failures/failure_cases.json")


In [ ]:
os.chdir(WORKDIR)

# External reference table: copied from the DENTEX 2023 challenge report (arXiv:2305.19112, Table II).
# Different test set (250 hidden challenge images) and different training budget -> context only.
EXTERNAL = [
    ("Hamamci I. (HierarchicalDet, baseline)", 0.691, 37.6, 60.2, 44.0),
    ("Mei S. (YOLOrtho)", 0.5972, 37.44, 56.72, 45.49),
    ("He L. (DINO + U-Net + YOLOv8 + WBF)", 0.5921, 37.06, 54.31, 44.77),
    ("Chen S. (DiffusionDet)", 0.644, 37.0, 58.1, 42.6),
    ("Choi K. (DETDet)", 0.7585, 36.48, 57.30, 42.84),
    ("Carion N. (DETR reference)", 0.514, 33.4, 52.8, 41.7),
    ("Ren S. (Faster R-CNN reference)", 0.533, 33.2, 54.3, 38.0),
    ("Lin T.Y. (RetinaNet reference)", 0.587, 32.5, 54.2, 35.6),
]
ext = pd.DataFrame(EXTERNAL, columns=["entry", "AR@100", "AP", "AP50", "AP75"]).set_index("entry")
print("EXTERNAL - DENTEX 2023 diagnosis leaderboard (250-image hidden test, percentages as published):")
display(ext)

print("\nOURS - same columns, measured above on the 50-image public validation split:")
display(summary[["AR@100", "AP", "AP50", "AP75"]].round(4))

print("\nNot a head-to-head comparison: different test images, 635 vs 705 training images, different schedules.")

best = summary["AP50"].idxmax()
rep = reports[best]
o = rep["coco_map"]["overall"]
sel = rep["threshold_selection"]
tau = sel.get("chosen_threshold", sel.get("best_threshold"))
print("\nMarkdown row for REPORT.md, generated from this run's JSON:")
print("| Run | AP | AP50 | AP75 | AR@100 | tau | val macro-F1 | test macro-F1 |")
print(f"| {best} | {o['map_50_95']:.4f} | {o['map_50']:.4f} | {o['map_75']:.4f} | "
      f"{o.get('ar_100', float('nan')):.4f} | {tau:.2f} | {float(sel.get('best_val_macro_f1')):.4f} | "
      f"{float(rep['threshold_metrics']['macro_f1']):.4f} |")


In [ ]:
os.chdir(WORKDIR)
import shutil, zipfile

zip_path = Path("viscan_benchmark_results.zip")
members = []
for folder in ("evaluation_frcnn", "evaluation_frcnn_tta", "evaluation_retinanet",
               "evaluation_retinanet_tta", "evaluation_ensemble", "evaluation_ensemble_tta"):
    for p in Path(folder).rglob("*"):
        if p.is_file():
            members.append((p, str(p)))
for p in ("runs/retinanet/loss_curve.png", "runs/retinanet/val_ap_curve.png",
          "checkpoints/retinanet/best_model.pth", "configs/retinanet.yaml"):
    if Path(p).exists():
        members.append((Path(p), Path(p).name))
for p in ("preds/frcnn", "preds/frcnn_tta", "preds/retinanet", "preds/retinanet_tta"):
    for f in Path(p).glob("predictions_*.json"):
        members.append((f, str(f)))
for log in Path(".").glob("eval_*.log"):
    members.append((log, log.name))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for src, arc in members:
        zf.write(src, arc)
print(f"zip: {zip_path.resolve()} ({zip_path.stat().st_size / 1024**2:.1f} MB, {len(members)} files)")

for name in ("evaluation_frcnn", "evaluation_retinanet", "evaluation_ensemble"):
    src = Path(name) / "final_evaluation_report.json"
    if src.exists():
        shutil.copy2(src, Path("/kaggle/working") / f"{name}.json")
        print("copied", src.name, "->", f"/kaggle/working/{name}.json")


## Decision rule for REPORT.md

Read the measured table above before writing anything.

1. **Architecture (E1):** keep Faster R-CNN deployed unless RetinaNet's test AP50 or AP is higher. If RetinaNet
   loses, report it as a measured negative result — the brief accepts that.
2. **TTA (E2):** adopt only if AP50 or AP improves. TTA runs two forward passes, so latency roughly doubles;
   `cpu_latency_ms` in the JSON is single-pass.
3. **WBF (E3):** a benchmark-level win only. Deployment requires the fusion latency to be acceptable; the
   ensemble script does not measure it (fusion wall-clock is the sum of member latencies).
4. Copy the generated markdown row into `REPORT.md`, keep the Faster R-CNN numbers that were measured earlier,
   and state which rows were re-measured in this session versus copied from the earlier report.
5. Do not compare our AP50 to the DENTEX leaderboard as if it were the same test set. `RESEARCH_BENCHMARKS.md`
   §5 explains exactly why.


## Optional cleanup

Run only after the zip and report copies exist. Removes the raw dataset archives to keep the Kaggle output
small; re-running the notebook afterwards re-downloads DENTEX.


In [ ]:
os.chdir(WORKDIR)
import shutil

assert Path("viscan_benchmark_results.zip").exists(), "build the zip before cleanup"
for p in ("data/dentex/zips", "data/dentex/extracted_train", "data/dentex/extracted_val"):
    target = Path(p)
    if target.exists():
        shutil.rmtree(target)
        print("removed", target)

print("final /kaggle/working size:",
      round(sum(f.stat().st_size for f in Path("/kaggle/working").rglob("*") if f.is_file()) / 1024**3, 2),
      "GB")
